## Example 1: tiktoken functionality test 1

In [ ]:
import tiktoken

# Example 1: Basic tokenization with different encodings
print("Basic tiktoken tokenization:")

# Get the cl100k_base encoding (used by GPT-3.5 and GPT-4)
enc = tiktoken.get_encoding("cl100k_base")
print(f"Encoding: {enc.name}")

# Test basic tokenization
text = "Hello, world! This is a test sentence."
tokens = enc.encode(text)
decoded = enc.decode(tokens)

print(f"Original text: '{text}'")
print(f"Token count: {len(tokens)}")
print(f"Tokens: {tokens[:10]}...")  # Show first 10 tokens
print(f"Decoded text: '{decoded}'")
print(f"Round-trip successful: {text == decoded}")

# Show available encodings
available_encodings = tiktoken.list_encoding_names()
print(f"Available encodings: {available_encodings}")

## Example 2: tiktoken functionality test 2

In [ ]:
import tiktoken

# Example 2: Model-specific tokenization and encoding comparison
print("Model-specific tokenization:")

# Test different model encodings
models = ["gpt-3.5-turbo", "gpt-4", "text-davinci-003"]
test_text = "The quick brown fox jumps over the lazy dog. 🦊🐕"

for model in models:
    try:
        enc = tiktoken.encoding_for_model(model)
        tokens = enc.encode(test_text)
        print(f"{model:20}: {len(tokens)} tokens")
        print(f"  Encoding: {enc.name}")
        print(f"  First 5 tokens: {tokens[:5]}")
    except KeyError:
        print(f"{model:20}: Model encoding not available")

# Compare encoding efficiency
texts = [
    "Short text",
    "This is a medium-length sentence with some technical terms like API, JSON, and HTTP.",
    "Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation."
]

enc = tiktoken.get_encoding("cl100k_base")
print(f"\nEncoding efficiency comparison:")
for i, text in enumerate(texts, 1):
    tokens = enc.encode(text)
    ratio = len(tokens) / len(text) if text else 0
    print(f"Text {i}: {len(text)} chars → {len(tokens)} tokens (ratio: {ratio:.3f})")

## Example 3: tiktoken functionality test 3

In [ ]:
import tiktoken

# Example 3: Advanced tokenization features and special tokens
print("Advanced tokenization features:")

# Get encoding with special tokens support
enc = tiktoken.get_encoding("cl100k_base")

# Test tokenization with special characters and unicode
special_texts = [
    "Hello 世界! 🌍",  # Mixed languages and emoji
    "Code: `print('hello')` and math: x²+y²=z²",  # Code and mathematical symbols
    "Email: user@example.com, URL: https://tiktoken.ai",  # Structured data
    "Numbers: 123,456.789 and dates: 2024-01-15",  # Numeric data
]

print("Special character handling:")
for text in special_texts:
    tokens = enc.encode(text)
    decoded = enc.decode(tokens)
    print(f"'{text[:30]}...' → {len(tokens)} tokens")
    print(f"  Round-trip: {'✓' if text == decoded else '✗'}")

# Test encoding with different disallowed special tokens handling
test_text = "This is a test with <|endoftext|> special token"
# By default tiktoken REFUSES text that spells a special token (ValueError), so input text cannot smuggle one in
try:
    enc.encode(test_text)
except ValueError as e:
    print(f"\nDefault encoding refuses the special token, as documented: {str(e).splitlines()[0]}")
else:
    raise AssertionError("tiktoken encoded '<|endoftext|>' by default; it should refuse it")

# With the check switched off, the same text is encoded as ordinary text
tokens_as_text = enc.encode(test_text, disallowed_special=())
print(f"As plain text: {len(tokens_as_text)} tokens")
assert enc.eot_token not in tokens_as_text

# Allow special tokens
try:
    tokens_allowed = enc.encode(test_text, allowed_special="all")
    print(f"With allowed special: {len(tokens_allowed)} tokens")
    assert enc.eot_token in tokens_allowed, "the special token was not encoded as its own id"
except Exception as e:
    print(f"Allowed special error: {e}")
    raise

print(f"\nEncoding object methods: {[m for m in dir(enc) if not m.startswith('_')]}")



## Example 4: tiktoken functionality test 4

In [ ]:
import tiktoken

# Example 4: Token counting and text splitting for AI applications
print("Token counting and text splitting:")

# Create a token counter utility
class TokenCounter:
    def __init__(self, model="gpt-3.5-turbo"):
        self.enc = tiktoken.encoding_for_model(model)
        self.model = model
    
    def count_tokens(self, text):
        """Count tokens in text"""
        return len(self.enc.encode(text))
    
    def split_text_by_tokens(self, text, max_tokens=100):
        """Split text into chunks with maximum token count"""
        tokens = self.enc.encode(text)
        chunks = []
        
        for i in range(0, len(tokens), max_tokens):
            chunk_tokens = tokens[i:i + max_tokens]
            chunk_text = self.enc.decode(chunk_tokens)
            chunks.append(chunk_text)
        
        return chunks

# Test token counting
counter = TokenCounter()

# Sample texts of different lengths
texts = [
    "Short message",
    "This is a medium-length paragraph that contains several sentences. It's designed to test how tiktoken handles moderately sized text inputs with various punctuation and formatting.",
    """This is a much longer text that simulates a document or article. It contains multiple paragraphs, various punctuation marks, and different types of content. This type of text is common in applications where you need to process large amounts of text data, such as summarization, translation, or analysis tasks. The goal is to understand how tokenization works with substantial content."""
]

print("Token counting results:")
for i, text in enumerate(texts, 1):
    token_count = counter.count_tokens(text)
    words = len(text.split())
    print(f"Text {i}: {len(text)} chars, {words} words, {token_count} tokens")
    print(f"  Tokens/word ratio: {token_count/words:.2f}")

# Test text splitting
long_text = " ".join([f"Sentence {i} contains some example content." for i in range(1, 21)])
chunks = counter.split_text_by_tokens(long_text, max_tokens=50)

print(f"\nText splitting (max 50 tokens per chunk):")
print(f"Original: {counter.count_tokens(long_text)} tokens")
print(f"Split into {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
    chunk_tokens = counter.count_tokens(chunk)
    print(f"  Chunk {i+1}: {chunk_tokens} tokens - '{chunk[:40]}...'")



## Example 5: tiktoken functionality test 5

In [ ]:
import tiktoken

# Example 5: Production patterns for AI/LLM applications
class AITextProcessor:
    """Production-ready text processor for AI applications"""
    
    def __init__(self, model="gpt-3.5-turbo", max_context=4096):
        self.enc = tiktoken.encoding_for_model(model)
        self.model = model
        self.max_context = max_context
        self.system_prompt_tokens = 0
    
    def set_system_prompt(self, prompt):
        """Set system prompt and calculate its token usage"""
        self.system_prompt_tokens = len(self.enc.encode(prompt))
        return self.system_prompt_tokens
    
    def validate_input(self, text):
        """Validate if text fits within context window"""
        tokens = len(self.enc.encode(text))
        available_tokens = self.max_context - self.system_prompt_tokens
        
        return {
            'tokens': tokens,
            'max_tokens': self.max_context,
            'available_tokens': available_tokens,
            'fits': tokens <= available_tokens,
            'utilization': tokens / available_tokens if available_tokens > 0 else float('inf')
        }
    
    def prepare_batch(self, texts, max_batch_tokens=8000):
        """Prepare batch of texts for processing"""
        batches = []
        current_batch = []
        current_tokens = self.system_prompt_tokens
        
        for text in texts:
            text_tokens = len(self.enc.encode(text))
            
            if current_tokens + text_tokens > max_batch_tokens:
                if current_batch:
                    batches.append(current_batch)
                current_batch = [text]
                current_tokens = self.system_prompt_tokens + text_tokens
            else:
                current_batch.append(text)
                current_tokens += text_tokens
        
        if current_batch:
            batches.append(current_batch)
        
        return batches

print("Production AI text processing:")

# Initialize processor
processor = AITextProcessor("gpt-4", max_context=8192)

# Set system prompt
system_prompt = "You are a helpful AI assistant that analyzes text content."
system_tokens = processor.set_system_prompt(system_prompt)
print(f"System prompt: {system_tokens} tokens")

# Test input validation
test_inputs = [
    "Short query",
    "Medium length query with several sentences that contains more detailed information about the task.",
    " ".join([f"Very long input sentence {i}." for i in range(100)])  # Very long input
]

print(f"\nInput validation:")
for i, text in enumerate(test_inputs, 1):
    validation = processor.validate_input(text)
    status = "✓ FITS" if validation['fits'] else "✗ TOO LONG"
    print(f"Input {i}: {validation['tokens']} tokens, {validation['utilization']:.1%} utilization - {status}")

# Test batch processing
documents = [f"Document {i} content with some text to analyze." for i in range(20)]
batches = processor.prepare_batch(documents, max_batch_tokens=200)

print(f"\nBatch processing:")
print(f"Documents: {len(documents)}")
print(f"Batches created: {len(batches)}")
for i, batch in enumerate(batches):
    batch_tokens = sum(len(processor.enc.encode(doc)) for doc in batch) + processor.system_prompt_tokens
    print(f"  Batch {i+1}: {len(batch)} docs, ~{batch_tokens} tokens")

print(f"\nProduction considerations:")
print("- Context window management")
print("- Token counting for cost estimation")
print("- Batch optimization for throughput")
print("- Input validation and truncation")